In [1]:
from dotenv import load_dotenv
import os
from hera.workflows import models, CronWorkflow, script, Artifact, Parameter, DAG, Steps, Step, NoneArchiveStrategy, Workflow, Task, WorkflowStatus
from hera.shared import global_config

load_dotenv("/home/otto/zarr_workflows/.env")

True

In [2]:
global_config.host = "https://services.eodc.eu/workflows/"
global_config.namespace = "s1sig0"
global_config.token = os.getenv("argo_token_prod")
global_config.image = "ghcr.io/oscipal/image_zarr:latest"

In [3]:
nfs_volume = [models.Volume(
    name="eodc-mount",
    persistent_volume_claim={"claimName": "eodc-nfs-claim"},
    )]

security_context = {"runAsUser": 75000,
                    "runAsGroup": 60028}

In [4]:
@script(volume_mounts=[models.VolumeMount(name="eodc-mount", mount_path="/eodc")])

def get_timerange():
    """
        Function to get a timerange from a yaml file with keys ranges, processing and done. Timerange is taken from ranges and moved to processing.
    """
    import yaml

    # Function to read and move the timerange
    def read_yaml(filename):
        with open(filename) as f:
            data = yaml.safe_load(f) # read the yaml
        popped = data["ranges"].pop(0) # Get the first value in ranges and delete it from ranges
        data["processing"].append(popped) # Append the timerange to processing
        with open(filename, "w") as f:
            yaml.dump(data, f) # Write the updated timeranges to the yaml
        return popped

    time_range = read_yaml("/eodc/private/tempearth/s1sig0_timesteps.yaml") # Use the function
    print(time_range) # Print it to be able to use it in following steps

In [5]:
@script(volume_mounts=[models.VolumeMount(name="eodc-mount", mount_path="/eodc")])

def end(time_range):
    """
        Function to be used at the end of a workflow. Takes a timerange and moves it from processing to done in a yaml.
    """
    import yaml

    # Function to move the timerange in the yaml, target is the timerange to be moved
    def read_and_move_yaml(filename, target):
            with open(filename) as f:
                data = yaml.safe_load(f) # read the yaml

            processing = data["processing"] # Get the values in the processing key

            if target not in processing:
                raise ValueError(f"{target!r} not found in ranges") # Raise an error if specified timerange is not there
            
            processing.remove(target) # Remove the specified timerange

            data["done"].append(target) # Append to done

            with open(filename, "w") as f:
                yaml.safe_dump(data, f, sort_keys=False) # Write the updated timeranges to the yaml

            return target

    read_and_move_yaml("/eodc/private/tempearth/s1sig0_timesteps.yaml", time_range) # Use the function

In [6]:
@script(volume_mounts=[models.VolumeMount(name="eodc-mount", mount_path="/eodc")])

def extend_and_get_time():
    """
        Function to extend the timedimension of the sentinel1-sig0 zarr store to current date. Also prints a timerange from end of existing zarr store to now, which can be used to write to the store.
    """

    import datetime
    import numpy as np
    import zarr

    # Get the origin date and the current date in the correct format (as days)
    now = datetime.datetime.now()
    now_np = np.datetime64(now).astype('datetime64[D]')
    origin = np.datetime64("2014-10-01T00:00:00").astype("datetime64[D]")

    # Get the new shape and extent of the timedimension of the zarr store, extent has to be in days since origin (int format)
    new_shape = int((now_np-origin).astype(int))
    new_extent = np.arange(0,new_shape,1)

    # Open the zarr store and group
    store = zarr.storage.LocalStore("/eodc/products/eodc/sentinel1_sig0/s1sig0.zarr")
    group = zarr.group(store=store)

    # Get the names of all arrays in the group which are dependant on the timedimension and need to be reshaped accordingly
    array_names=set(group.array_keys()) # Get all array names
    coords = {"time", "x", "y", "spatial_ref", "relative_orbit_number"} # Define the names of the coordinates, and arrays not dependant on the time dimension
    data_arrays = array_names-coords # Get only array names which need to be reshaped

    # Reshape all array names to the new timedimension shape
    group["time"].resize(new_shape) # Reshape the time dimesion
    for array in data_arrays:
        group_shape  = group[array].shape # Get the current shape
        group[array].resize((group_shape[0], new_shape, group_shape[2], group_shape[3])) # Reshape only the time axis

    # Consolidate metadata to ensure proper reading
    zarr.consolidate_metadata(store)

    # Reopen the zarr store and group with updated shapes
    store = zarr.storage.LocalStore("/eodc/products/eodc/sentinel1_sig0/s1sig0.zarr")
    group = zarr.group(store=store)

    # Write the new time extent to the time array
    group["time"][:]=new_extent

    # Get the timerange and pass it to be used in writing to the zarr store steps
    time_range = str(origin + group["time"][:].max())+"/"+str(now_np)
    print(time_range)

In [7]:
@script(volume_mounts=[models.VolumeMount(name="eodc-mount", mount_path="/eodc")])

def write_data(tile: str, time_range):
    """
        Function the write data to a zarr store. Uses data from the EODC stac catalogue and needs a EQUI7 Tile and timerange as input to query the catalogue
    """

    import pystac_client as pc
    import xarray as xr
    import zarr
    import numpy as np
    import rioxarray
    import pandas as pd
    from datetime import datetime
    from collections import defaultdict

    def group_by_relative_orbit(items, key="sat:relative_orbit"):
        """
            Function to group a list of stac items by their relative orbit. Result is a dictionaray with rel orbit as keys and items as values.
        """
        groups = defaultdict(list)
        for it in items:
            groups[it[0].properties[key]].append(it)
        return dict(groups)

    def get_idx(array1, array2):
        """
            Function to get the indices of coordinates from an array in a different array. To be used when wanting to write to a zarr store with indexing.
        """
        min = np.where(array1==array2[0])[0][0]
        max = np.where(array1==array2[-1])[0][0]+1
        return min, max

    def load_data(item, pols):
        """
            Load an item and add the sensing date as a time dimension, also rename acoording the polarization. If multiple polarizations are to be used a merges dataset will be created.
        """
        if type(pols)==str:
            data = rioxarray.open_rasterio(item.assets[pols].href).compute().expand_dims(time=pd.to_datetime([item.properties["datetime"]]).tz_convert(None)).rename(pols)
        else:
            data = []
            for pol in pols:
                data.append(rioxarray.open_rasterio(item.assets[pol].href).compute().expand_dims(time=pd.to_datetime([item.properties["datetime"]]).tz_convert(None)).rename(pol))
            
            data = xr.merge(data)
        return data.squeeze()

    def get_datetime(item):
        """
            Get the datetime from an item in the correct format
        """
        return datetime.strptime(item.properties["datetime"], "%Y-%m-%dT%H:%M:%SZ")

    def group_dates(item_list):
        """
            Group stac items dependant on their sensing dates. If sensing dates are close enough so the items are from the same orbit the will be grouped in a list of lists.
        """
        
        # Initialize list in list
        grouped_items = [[]]
        i=0


        for item in item_list:
            
            # If list in grouped items is empty append the item
            if not grouped_items[i]:
                grouped_items[i].append(item)
            
            # Otherwise check if the item is close enough to the existing item in the list and append if thats the case, if not append a new list to the grouped items with that item
            else: 
                if get_datetime(item) - get_datetime(grouped_items[i][-1]) <= pd.Timedelta(seconds=100):
                    grouped_items[i].append(item)

                else:
                    grouped_items.append([item])
                    i+=1

        return grouped_items

    def read_and_merge_items(items, pols):
        """
            Use the function load_data on grouped items and combine them according to the group. Reads items and combines data with a close enough sensing date to a single array, then combines the arrays to a large dataset. Works for a single polarization or a list of polarizations to be read.
        """
        first = True

        if type(pols)==list:
            datasets = []
            for pol in pols:
                # Iterate through a group of items and load the data according to the polarization
                for item in items:
                    ds = load_data(item, pol)
                    
                    # If first data which is read, rename 
                    if first:
                        data = ds
                        first = False
                    
                    # Otherwise the nodata values of the existing array are overwritten with the new data
                    else:
                        data = xr.where(data==-9999, ds, data, keep_attrs=True)

                # append the combined array to a list of arrays, expand the time dim if not there already
                if "time" in data.dims:      
                    datasets.append(data)
                else:
                    datasets.append(data.expand_dims(time=pd.to_datetime([item.properties["datetime"]]).tz_convert(None)))

                # Reset and restart for the next group
                first=True

            # Merge the datasets to a large dataset
            data = xr.merge(datasets)

        # Works similar but only for one polarization
        else:
            for item in items:
                ds = load_data(item, pols)
                
                if first:
                    data = ds
                    first = False
                
                else:
                    data = xr.where(data==-9999, ds, data, keep_attrs=True)

            data = data.to_dataset(name=pols)

        # Return the squeezed dataset
        return data.squeeze()

    # Open the EODC stac catalogue
    pc_client = pc.Client.open("https://stac.eodc.eu/api/v1")
    
    # Query the catalogue according to tile and timerange passed in the Function definition
    search = pc_client.search(
        collections=["SENTINEL1_SIG0_20M"],
        datetime=time_range,
        query={"Equi7_TileID": {"eq": f"EU020M_{tile}T3"}})

    # Get the resulting items
    items_eodc = search.item_collection()

    # If items are existing
    if items_eodc:

        # Write items as a list and sort ascending by time
        item_list = list(items_eodc)[::-1]

        # Group the items according to their sensing date, similiar sensing dates (from the same satellite scene) are in their own list
        grouped_items = group_dates(item_list)

        # Open the zarr store
        store = zarr.storage.LocalStore("/eodc/products/eodc/sentinel1_sig0/s1sig0.zarr")
        group = zarr.group(store=store)
        
        # Get the x and y extent, also the relative orbits
        x_extent = group["x"][:]
        y_extent = group["y"][:]
        rel_orbit_extent = group["relative_orbit_number"][:]

        # Origin for the sensing time
        sensing_origin = np.datetime64("2014-10-01T00:00:00")

        # Get the start and end date from the timerange
        start = np.datetime64(time_range.split("/", 1)[0].strip(), "D")
        end = np.datetime64(time_range.split("/", 1)[1].strip(), "D")
        
        # Group the items by their relative orbit (each group in grouped_items has the same relative orbit)
        grouped_orbits = group_by_relative_orbit(grouped_items)

        # grouped_orbits is a dict with relative orbit as key and items as values
        for orbit, items_orbits in grouped_orbits.items():
            print(f"{orbit} started")

            # Get the index of the relative orbit in the zarr store to know where to write to
            orbit_index = np.where(rel_orbit_extent==orbit)[0][0]
            
            # Initialize list to write data to 
            datasets_orbits = []

            # items_orbits contains groups of items according to their sensing dates. Each of those groups are read and merged into one dataset with a single time coordinate
            for items in items_orbits:

                # Try and read and merge data for both polarizations, if only one polarization is available catch a possible error and only use on polarization
                try:
                    ds = read_and_merge_items(items, ["VV", "VH"])
                except KeyError:
                    try:
                        ds = read_and_merge_items(items, ["VV"])
                    except KeyError:
                        ds = read_and_merge_items(items, ["VH"])
                
                # Get the relative orbit number to be a dimesion
                ds = ds.expand_dims({"rel_orbit_number": [ds.attrs["rel_orbit_number"]]})

                # The accurate sensing date and absolute orbit number will be data variables
                ds["sensing_date"] = (ds['time'].values.astype("datetime64[s]") - sensing_origin).astype("int64")
                ds["abs_orbit_number"] = ds.attrs["abs_orbit_number"]
                
                # The time dimension will be in only be stored without the time. Only Day of sensing is used 
                ds['time'] = ds['time'].astype('datetime64[D]')

                # Append the read dataset to a list of datasets
                datasets_orbits.append(ds)
                ds = None

            # Concat the datasets
            combined_orbits = xr.concat(datasets_orbits, dim="time", combine_attrs="override")

            # If some time steps are missing they are created and filled with nodata values
            full_times = pd.date_range(start=start, end=end, freq='D')
            result = combined_orbits.reindex(time=full_times, fill_value=-9999)

            # Transpose for further usage
            result = result.transpose("rel_orbit_number", "time", "y", "x")

            # The accurate sensing times and absolute orbit numbers will be a seperate array in the zarr store and have the same shape as the actual data, so they have to have the same dimension as the data
            sensing_dates = result["sensing_date"].values.reshape(1,result.sizes["time"],1,1)
            abs_orbit_numbers = result["abs_orbit_number"].values.reshape(1,result.sizes["time"],1,1)

            # X and Y are the center of a pixel in the current dataset, this is changed to match the input data
            result["x"] = result.x-10
            result["y"] = result.y+10

            # Get the x and y index of where to write the data to in the zarr store
            x_min, x_max = get_idx(x_extent, result["x"].values)
            y_min, y_max = get_idx(y_extent, result["y"].values)

            # Get the time index of where to write the data to in the zarr store
            time_origin = np.datetime64("2014-10-01")
            time_min = (result.time.min().values.astype("datetime64[D]") - time_origin).astype("int64")
            time_max = (result.time.max().values.astype("datetime64[D]") - time_origin).astype("int64")+1

            # Write the data to the zarr store
            group["VH"][orbit_index:orbit_index+1,time_min:time_max, y_min:y_max, x_min:x_max] = result["VH"].values
            group["VV"][orbit_index:orbit_index+1,time_min:time_max, y_min:y_max, x_min:x_max] = result["VV"].values

            # Reshape the sensing times and absolute orbit numbers to the size the should be
            sensing_dates = np.broadcast_to(sensing_dates, (1,time_max-time_min, y_max-y_min, x_max-x_min))
            abs_orbit_numbers = np.broadcast_to(abs_orbit_numbers, (1,time_max-time_min, y_max-y_min, x_max-x_min))

            # and write them to the zarr store
            group["sensing_date"][orbit_index:orbit_index+1,time_min:time_max, y_min:y_max, x_min:x_max] = sensing_dates
            group["absolute_orbit_number"][orbit_index:orbit_index+1,time_min:time_max, y_min:y_max, x_min:x_max] = abs_orbit_numbers
            print(f"{orbit} done")
        print("success")

    else:
        print("no items in collection")

In [8]:
# Generate a CronWorkflow for the backlog
# First step is to get the timerange from a yaml file and use it in the writing process
# Secondly the data is written for each tile subsequently
# Last step is to log that the timerange is done
with CronWorkflow(
    generate_name="s1sig0-zarr-backlog-",
    schedule = "0 */2 * * *",
    volumes = nfs_volume,
    security_context=security_context,
    entrypoint="workflow"
) as w_backlog:
    with DAG(name="workflow"):
        
        tr = get_timerange()
        process1 = write_data(name="E045N015", arguments={"tile":"E045N015", "time_range": tr.result})
        process2 = write_data(name="E048N015", arguments={"tile":"E048N015", "time_range": tr.result})
        process3 = write_data(name="E051N015", arguments={"tile":"E051N015", "time_range": tr.result})
        process4 = write_data(name="E048N012", arguments={"tile":"E048N012", "time_range": tr.result})
        process5 = write_data(name="E051N012", arguments={"tile":"E051N012", "time_range": tr.result})
        final = end(arguments={"time_range": tr.result})

        tr >> process1 >> process2 >> process3 >> process4 >> process5 >> final

In [9]:
w_backlog.create()

CronWorkflow(api_version=None, kind=None, metadata=ObjectMeta(annotations=None, cluster_name=None, creation_timestamp=Time(__root__=datetime.datetime(2025, 8, 26, 19, 29, 55, tzinfo=datetime.timezone.utc)), deletion_grace_period_seconds=None, deletion_timestamp=None, finalizers=None, generate_name='s1sig0-zarr-backlog-', generation=1, labels={'workflows.argoproj.io/creator': 'system-serviceaccount-default-jenkins'}, managed_fields=[ManagedFieldsEntry(api_version='argoproj.io/v1alpha1', fields_type='FieldsV1', fields_v1=FieldsV1(), manager='argo', operation='Update', subresource=None, time=Time(__root__=datetime.datetime(2025, 8, 26, 19, 29, 55, tzinfo=datetime.timezone.utc)))], name='s1sig0-zarr-backlog-rnfg7', namespace='s1sig0', owner_references=None, resource_version='50337505', self_link=None, uid='ed0ce8ce-1719-412f-96c4-4a185cc07cf3'), spec=CronWorkflowSpec(concurrency_policy=None, failed_jobs_history_limit=None, schedule='0 */2 * * *', starting_deadline_seconds=None, successful_

In [ ]:
# Generate a CronWorkflow for updating the zarr store to run once a day
# First step is to extend the time dimension from the zarr store and get the timerange
# Secondly the data is written for each tile subsequently
with CronWorkflow(
    generate_name="s1sig0-zarr-update-",
    schedule = "0 5 * * *",
    volumes = nfs_volume,
    security_context=security_context,
    entrypoint="workflow"
) as w_update:
    with DAG(name="workflow"):
        
        tr = extend_and_get_time()
        process1 = write_data(name="E045N015", arguments={"tile":"E045N015", "time_range": tr.result})
        process2 = write_data(name="E048N015", arguments={"tile":"E048N015", "time_range": tr.result})
        process3 = write_data(name="E051N015", arguments={"tile":"E051N015", "time_range": tr.result})
        process4 = write_data(name="E048N012", arguments={"tile":"E048N012", "time_range": tr.result})
        process5 = write_data(name="E051N012", arguments={"tile":"E051N012", "time_range": tr.result})

        tr >> process1 >> process2 >> process3 >> process4 >> process5

In [ ]:
w_updated.create()